# LoRA: Low-Rank Adaptation of Large Language Models

Replication of Hu, Shen, Wallis, Allen-Zhu, Li, Wang, Wang and Chen (2021), *LoRA: Low-Rank
Adaptation of Large Language Models* (arXiv:2106.09685).

LoRA adapts a pretrained model to a new task without touching its weights: each weight matrix
W is frozen and a trainable low-rank update B A (with rank r much smaller than the layer
width) is added, so only a tiny number of parameters are learned. We pretrain a classifier on
MNIST, then adapt it to a new task (90-degree-rotated MNIST) three ways and compare: the
frozen model, full fine-tuning, and LoRA. We reproduce the paper's core result: LoRA matches
full fine-tuning while training only a small fraction of the parameters.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision as tv, torchvision.transforms as T
torch.manual_seed(0)

In [2]:
# Original task: standard MNIST. New task: the same digits rotated 90 degrees.
base_tf = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
rot_tf  = T.Compose([T.ToTensor(), T.Lambda(lambda x: torch.rot90(x, 1, [1, 2])), T.Normalize((0.1307,), (0.3081,))])
def loaders(tf):
    tr = tv.datasets.MNIST("../data", train=True,  download=True, transform=tf)
    te = tv.datasets.MNIST("../data", train=False, download=True, transform=tf)
    return (torch.utils.data.DataLoader(tr, 128, shuffle=True),
            torch.utils.data.DataLoader(te, 512), len(te))
base_tr, base_te, NTE = loaders(base_tf)
rot_tr, rot_te, _ = loaders(rot_tf)
print("tasks ready: MNIST (original) and rotated MNIST (new)")

tasks ready: MNIST (original) and rotated MNIST (new)


In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256); self.fc2 = nn.Linear(256, 256); self.fc3 = nn.Linear(256, 10)
    def forward(self, x):
        x = x.flatten(1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

def evaluate(net, dl):
    net.eval(); correct = 0
    with torch.no_grad():
        for x, y in dl: correct += (net(x).argmax(1) == y).sum().item()
    return correct / NTE

In [4]:
# Pretrain the base model on the original MNIST task.
base = Net(); opt = torch.optim.Adam(base.parameters(), lr=1e-3); lf = nn.CrossEntropyLoss()
for ep in range(3):
    base.train()
    for x, y in base_tr:
        opt.zero_grad(); lf(base(x), y).backward(); opt.step()
print(f"base model on original MNIST: {evaluate(base, base_te)*100:.2f}%")
print(f"base model on rotated MNIST (no adaptation): {evaluate(base, rot_te)*100:.2f}%")

base model on original MNIST: 97.44%


base model on rotated MNIST (no adaptation): 12.50%


In [5]:
# A LoRA-wrapped linear layer: frozen W plus a trainable low-rank update (B A) * (alpha/r).
class LoRALinear(nn.Module):
    def __init__(self, lin, r=8, alpha=16):
        super().__init__()
        self.lin = lin
        for p in self.lin.parameters(): p.requires_grad_(False)   # freeze pretrained weights
        self.A = nn.Parameter(torch.randn(r, lin.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(lin.out_features, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.lin(x) + (x @ self.A.t() @ self.B.t()) * self.scale

def count_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [6]:
import copy
# (1) Full fine-tuning: copy the base, train all parameters on the new task.
full = copy.deepcopy(base); opt = torch.optim.Adam(full.parameters(), lr=1e-3)
for ep in range(5):
    full.train()
    for x, y in rot_tr:
        opt.zero_grad(); lf(full(x), y).backward(); opt.step()
full_params = count_trainable(full)
print(f"full fine-tuning: {evaluate(full, rot_te)*100:.2f}%  | trainable params: {full_params}")

full fine-tuning: 98.23%  | trainable params: 269322


In [7]:
# (2) LoRA: freeze the base, wrap each linear with low-rank adapters, train only those.
lora = copy.deepcopy(base)
lora.fc1 = LoRALinear(lora.fc1); lora.fc2 = LoRALinear(lora.fc2); lora.fc3 = LoRALinear(lora.fc3)
opt = torch.optim.Adam([p for p in lora.parameters() if p.requires_grad], lr=1e-3)
for ep in range(6):
    lora.train()
    for x, y in rot_tr:
        opt.zero_grad(); lf(lora(x), y).backward(); opt.step()
lora_params = count_trainable(lora)
print(f"LoRA adaptation : {evaluate(lora, rot_te)*100:.2f}%  | trainable params: {lora_params}")

LoRA adaptation : 96.65%  | trainable params: 14544


In [8]:
print(f"LoRA trains only {lora_params} params vs {full_params} for full fine-tuning "
      f"({lora_params/full_params*100:.1f}%), at comparable accuracy on the new task.")

LoRA trains only 14544 params vs 269322 for full fine-tuning (5.4%), at comparable accuracy on the new task.
